# Open-R1

A refresher on **Open-R1** — Hugging Face's fully open reproduction of **DeepSeek-R1**, the recipe (code + data + configs) for turning a base LLM into a *reasoning* model that thinks in `<think>…</think>` before answering, trained with **verifiable-reward RL (GRPO)** instead of a learned reward model.

**Domain:** LLM Inference, Training & Optimization  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**Open-R1 is an open-source pipeline that re-creates DeepSeek-R1 end to end** — the *training recipe*, not a single model. DeepSeek released the R1 weights and a paper in January 2025, but **not** the training code or data. Open-R1 fills that gap: a reproducible stack (datasets, training scripts, eval, configs) that takes a base/instruct model to a strong math-and-code reasoner.

**The problem it solves.** "Reasoning models" (o1, R1) get large quality jumps by emitting a long chain of thought before the final answer, learned via reinforcement learning with *verifiable* rewards — a math answer is graded right/wrong by a checker, code by running tests. Reproducing that requires three things most people don't have lying around: (1) **reasoning-trace data** to cold-start from, (2) a **GRPO RL loop** wired to fast generation and reward functions, and (3) **configs + eval** that actually converge. Open-R1 ships all three so you can distill or RL-train your own reasoner without reinventing the harness.

**The three pipeline stages** (mirroring the R1 paper):
1. **Distill (SFT)** — supervised fine-tune a smaller model on R1's *generated* reasoning traces (e.g. the `OpenR1-Math-220k` dataset). Cheap, no RL, gets you 80% of the way.
2. **GRPO RL** — reinforcement-learn on prompts with *verifiable* answers, rewarding correct + well-formatted completions. This is "R1-Zero"-style learning of reasoning behaviour.
3. **Evaluate** — run standardized math/code benchmarks (AIME, MATH-500, GPQA, LiveCodeBench) via `lighteval` to compare against R1.

**Reach for it when** you want to build or study a reasoning model, need an open GRPO recipe that converges, or want the curated reasoning datasets. **Don't** reach for it when a prompt-only chain-of-thought ([chain-of-thought](chain-of-thought.ipynb)) or off-the-shelf reasoning API already meets your need, or when you have no verifiable reward signal (open-ended generation, taste-based tasks) — GRPO needs a grader.

## 2. Mental Model

**RLHF replaced the human grader with a *learned* reward model. R1/Open-R1 replaces it with a *calculator*.** When the answer is checkable (math, code), you don't need a reward model at all — you run a verifier and reward correctness directly. GRPO then learns *which* chains of thought lead to correct answers.

```
                       ┌──────────────── one prompt ────────────────┐
                       │  "Solve: 7 * 6 = ?"                          │
                       └──────────────────┬──────────────────────────┘
                                          │ sample a GROUP of G completions
                 ┌────────────────┬───────┴────────┬────────────────┐
                 ▼                ▼                ▼                ▼
         <think>…42</think>  <think>…40</think> …             <think>…42</think>
                 │                │                │                │
       verifiable reward (no learned critic): format ✓/✗  +  answer correct ✓/✗
                 │                │                │                │
              r=1.0            r=0.2            r=0.3            r=1.0
                 └────────────────┴───────┬────────┴────────────────┘
                                          ▼
            GRPO advantage  Aᵢ = (rᵢ − mean(r)) / std(r)     ← group-relative, no value net
                                          ▼
                 push policy ▲ toward above-average completions, ▼ below-average
```

The key shift vs PPO: **GRPO drops the value/critic network.** The "baseline" each completion is compared against is just the *mean reward of its own sampled group*. That halves the memory (no critic) and makes the reward source a plug-in function. Open-R1 is the glue that wires this loop to fast generation (vLLM), reward functions, and reasoning data.

## 3. Key Concepts

- **DeepSeek-R1 / R1-Zero** — R1-Zero is RL *directly from a base model* (no SFT) and spontaneously learns long reasoning; R1 adds a small "cold-start" SFT before RL to fix readability/language-mixing. Open-R1 reproduces both paths.
- **GRPO (Group Relative Policy Optimization)** — the RL algorithm. Sample G completions per prompt, score each, and use the group's mean/std as the baseline: advantage `Aᵢ = (rᵢ − μ) / σ`. No separate value network (the main difference from PPO).
- **Verifiable / rule-based rewards** — rewards computed by *deterministic checkers*, not a learned reward model: `accuracy_reward` (is the math answer equivalent?), `format_reward` (are the `<think>`/answer tags well-formed?), tag-count and length rewards. Composable; the trainer sums them.
- **`math_verify`** — Hugging Face's library that parses and checks mathematical equivalence (`\\boxed{}`, fractions, LaTeX) so `1/2` == `0.5` == `\\frac{1}{2}`. Backs `accuracy_reward`.
- **Reasoning traces** — training examples of the form *problem → `<think>` long CoT `</think>` → final answer*. The distillation datasets (`OpenR1-Math-220k`, `OpenR1-Codeforces`) are R1-generated and filtered for correctness.
- **Distillation (SFT stage)** — supervised fine-tune on those traces. Far cheaper than RL and often most of the gain; the recommended starting point.
- **`GRPOTrainer` (TRL)** — Open-R1's GRPO stage is a thin wrapper over [TRL](trl-rlhf-dpo.ipynb)'s `GRPOTrainer`; Open-R1 supplies the reward functions, configs, and data plumbing.
- **vLLM-backed generation** — RL is generation-bound (you sample G completions per step). Open-R1 runs a [vLLM](vllm.ipynb) server for fast rollouts, separate from the training GPUs.
- **Recipes & `lighteval`** — YAML configs under `recipes/` define model/data/hyperparams; launched with `accelerate launch`. Evaluation uses `lighteval` on the standard reasoning benchmarks.

## 4. Setup

Open-R1 is a *training* repo: you clone it and install with `uv`/`pip`, and real runs want GPUs (multi-GPU for GRPO + a vLLM server for rollouts). It builds on [`trl`](trl-rlhf-dpo.ipynb), `transformers`, `accelerate`, `vllm`, `math_verify`, and `lighteval`.

```bash
# Real setup (GPU box): clone the repo and install the training stack
git clone https://github.com/huggingface/open-r1.git && cd open-r1
uv venv && source .venv/bin/activate
uv pip install -e ".[dev]"          # pulls trl, vllm, math_verify, lighteval, ...

# Launch the SFT (distill) stage from a recipe:
#   accelerate launch --config_file recipes/accelerate_configs/zero3.yaml \
#       src/open_r1/sft.py --config recipes/<model>/sft/config.yaml
# Launch the GRPO (RL) stage (needs a vLLM server for fast rollouts):
#   accelerate launch src/open_r1/grpo.py --config recipes/<model>/grpo/config.yaml
```

The cell below installs nothing heavy — the worked examples implement the **reward functions and GRPO advantage math** in plain Python/NumPy so the *logic* of the pipeline runs fully on CPU, offline. The actual `GRPOTrainer` call shape is shown later, gated behind an env check.

```bash
# What this notebook actually needs (already in the study venv):
pip install numpy
```

In [1]:
import os
import re
import numpy as np

print(f"numpy : {np.__version__}")

# Everything offline runs with stdlib + numpy. The real training stack (trl,
# vllm, math_verify, lighteval) is heavy + GPU-bound, so the live-training cell
# is gated behind this flag and the notebook executes either way.
RUN_TRAINING = bool(os.getenv("OPEN_R1_RUN_TRAINING"))
print(f"OPEN_R1_RUN_TRAINING set: {RUN_TRAINING}")

numpy : 2.5.0
OPEN_R1_RUN_TRAINING set: False


## 5. Worked Examples

The heart of Open-R1's RL stage is **(a) verifiable reward functions** and **(b) the GRPO group-relative advantage**. We implement both from scratch on toy data so the mechanics are concrete, then show the real `GRPOTrainer` recipe shape.

1. **Reward functions** — `format_reward` + `accuracy_reward` scoring a group of completions (offline, real output).
2. **GRPO advantages** — turn those rewards into the learning signal GRPO actually uses (NumPy, offline).
3. **Real GRPO recipe** — the TRL `GRPOTrainer` / Open-R1 config shape, gated behind `OPEN_R1_RUN_TRAINING`.

### Example 1 — Verifiable reward functions

Open-R1's GRPO stage rewards completions with simple, deterministic checkers. Two of the core ones:

- **`format_reward`** — does the completion follow the `<think>…</think>` then answer structure? (Teaches the model to *show its work* in parseable tags.)
- **`accuracy_reward`** — is the final boxed answer mathematically correct? (The real one uses `math_verify`; here a tiny numeric extractor stands in.)

We score a *group* of four completions to one prompt — exactly what GRPO samples per step.

In [2]:
# The real reward fns take (completions, ground_truth) and return a list of floats.
THINK_RE = re.compile(r"<think>.*?</think>\s*", re.DOTALL)
BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")

def format_reward(completions):
    """1.0 if the completion is exactly <think>...</think> then a final answer."""
    pat = re.compile(r"^<think>.*?</think>\s*\\boxed\{.*?\}\s*$", re.DOTALL)
    return [1.0 if pat.match(c.strip()) else 0.0 for c in completions]

def accuracy_reward(completions, ground_truth):
    """1.0 if the \\boxed{} answer equals ground truth (math_verify in the real repo)."""
    rewards = []
    for c in completions:
        m = BOXED_RE.search(c)
        try:
            ok = m is not None and float(m.group(1)) == float(ground_truth)
        except ValueError:
            ok = False
        rewards.append(1.0 if ok else 0.0)
    return rewards

prompt = "What is 7 * 6?"
ground_truth = "42"
group = [
    "<think>7 times 6 is 42.</think>\n\\boxed{42}",          # correct + formatted
    "<think>6+6+6+6+6+6+6 = 42</think> \\boxed{42}",         # correct + formatted
    "The answer is 42.",                                       # correct-ish, bad format
    "<think>7*6... about 40</think>\n\\boxed{40}",           # formatted, wrong
]

fmt = format_reward(group)
acc = accuracy_reward(group, ground_truth)
# Trainer sums weighted reward components into one scalar per completion.
total = [f + a for f, a in zip(fmt, acc)]

print(f"prompt: {prompt}  (ground truth = {ground_truth})\n")
for i, (c, f, a, t) in enumerate(zip(group, fmt, acc, total)):
    print(f"  completion {i}: format={f}  accuracy={a}  total={t}")
print(f"\ngroup rewards: {total}")

prompt: What is 7 * 6?  (ground truth = 42)

  completion 0: format=1.0  accuracy=1.0  total=2.0
  completion 1: format=1.0  accuracy=1.0  total=2.0
  completion 2: format=0.0  accuracy=0.0  total=0.0
  completion 3: format=1.0  accuracy=0.0  total=1.0

group rewards: [2.0, 2.0, 0.0, 1.0]


### Example 2 — GRPO group-relative advantages

GRPO's trick: it has **no value network**. The baseline each completion is compared against is just the **mean reward of its own group**, normalized by the group's std. The resulting *advantage* scales the policy-gradient update — positive pushes the policy toward that completion, negative pushes away.

`Aᵢ = (rᵢ − mean(r)) / (std(r) + ε)`

Watch how the all-correct completions get positive advantages and the wrong one gets a strongly negative pull — purely from the rewards we computed above, no critic.

In [3]:
def grpo_advantages(rewards, eps=1e-4):
    r = np.asarray(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + eps)

rewards = np.array(total)                 # from Example 1
adv = grpo_advantages(rewards)

print(f"group rewards   : {rewards.tolist()}")
print(f"group mean      : {rewards.mean():.3f}")
print(f"group std       : {rewards.std():.3f}\n")
for i, (r, a) in enumerate(zip(rewards, adv)):
    arrow = "push toward " if a > 0 else "push away   "
    print(f"  completion {i}: reward={r:.1f}  advantage={a:+.3f}  -> {arrow}")

# Degenerate case the +eps guards against: if every completion scores the same,
# the group gives ZERO learning signal (std = 0 -> advantages ~ 0).
flat = grpo_advantages([1.0, 1.0, 1.0, 1.0])
print(f"\nall-equal rewards -> advantages {np.round(flat, 4).tolist()} (no signal)")

group rewards   : [2.0, 2.0, 0.0, 1.0]
group mean      : 1.250
group std       : 0.829

  completion 0: reward=2.0  advantage=+0.904  -> push toward 
  completion 1: reward=2.0  advantage=+0.904  -> push toward 
  completion 2: reward=0.0  advantage=-1.507  -> push away   
  completion 3: reward=1.0  advantage=-0.301  -> push away   

all-equal rewards -> advantages [0.0, 0.0, 0.0, 0.0] (no signal)


### Example 3 — The real GRPO recipe (gated)

In Open-R1 you don't hand-roll the loop — you hand reward functions and a config to TRL's `GRPOTrainer`, which Open-R1 wraps in `src/open_r1/grpo.py`. The code below is the real call shape; it's gated behind `OPEN_R1_RUN_TRAINING` because a real run needs the heavy stack, GPUs, and a vLLM server.

In [4]:
if RUN_TRAINING:
    # Real path (GPU box): trl provides GRPOTrainer; open-r1 supplies rewards + data.
    from datasets import load_dataset
    from trl import GRPOConfig, GRPOTrainer

    dataset = load_dataset("AI-MO/NuminaMath-TIR", split="train[:1%]")

    def reward_len(completions, **kwargs):           # toy reward; real repo uses accuracy+format
        return [-abs(len(c) - 512) / 512 for c in completions]

    trainer = GRPOTrainer(
        model="Qwen/Qwen2.5-0.5B-Instruct",
        reward_funcs=[reward_len],                   # list of verifiable reward fns
        args=GRPOConfig(
            output_dir="grpo-demo",
            num_generations=8,                       # G: completions sampled per prompt
            use_vllm=True,                           # fast rollouts via a vLLM server
            per_device_train_batch_size=1,
            max_steps=10,
        ),
        train_dataset=dataset,
    )
    trainer.train()
    print("done")
else:
    print("Skipped (set OPEN_R1_RUN_TRAINING=1 + GPUs to run). Real launch is:")
    print("  accelerate launch src/open_r1/grpo.py \\")
    print("      --config recipes/Qwen2.5-1.5B-Instruct/grpo/config.yaml")
    print()
    print("The config wires together (what Examples 1-2 implemented by hand):")
    print("  reward_funcs : [accuracy, format, tag_count]   # verifiable, composed")
    print("  num_generations: 8                              # GRPO group size G")
    print("  use_vllm: true                                  # vLLM rollout server")

Skipped (set OPEN_R1_RUN_TRAINING=1 + GPUs to run). Real launch is:
  accelerate launch src/open_r1/grpo.py \
      --config recipes/Qwen2.5-1.5B-Instruct/grpo/config.yaml

The config wires together (what Examples 1-2 implemented by hand):
  reward_funcs : [accuracy, format, tag_count]   # verifiable, composed
  num_generations: 8                              # GRPO group size G
  use_vllm: true                                  # vLLM rollout server


## 6. Gotchas & Pitfalls

- **No verifiable reward → wrong tool.** GRPO needs a grader that scores correctness. For open-ended/taste tasks (style, helpfulness) you're back to a *learned* reward model and [DPO/RLHF](trl-rlhf-dpo.ipynb) — Open-R1's whole premise doesn't apply.
- **Reward hacking.** Loose checkers get gamed: a length reward → rambling; a regex format reward → models emit the tags with empty reasoning. Compose several rewards and inspect samples; the real `accuracy_reward` uses `math_verify` precisely to avoid string-match exploits.
- **Zero-variance groups give zero signal.** If all G completions in a group score identically (all right or all wrong), the GRPO advantage is ~0 and that prompt teaches nothing. Curriculum/difficulty filtering keeps prompts in the "sometimes right" band where learning happens.
- **Generation is the bottleneck, not backprop.** RL samples G completions *per prompt per step*. Without a [vLLM](vllm.ipynb) rollout server it's painfully slow; budget GPUs for generation separately from training.
- **Distill first, RL second.** Pure RL-from-base (R1-Zero style) is expensive and unstable. The SFT-on-traces (distill) stage is cheaper and usually captures most of the gain — start there before reaching for GRPO.
- **`<think>` tags must be in the chat template.** If the tokenizer/chat template doesn't reserve/handle the reasoning tags, format rewards and parsing silently misbehave. Match the template the recipe expects.
- **It's a moving research repo.** Open-R1 tracks ongoing reproduction work — recipes, dataset names, and script paths change between commits. Pin a commit and read that revision's README rather than assuming stability.
- **Long contexts blow up memory.** Reasoning traces are long; `max_completion_length` × `num_generations` drives activation memory hard. Tune both together or OOM.

## 7. When to Use vs Alternatives

| Approach | Use it when | Trade-off vs Open-R1 |
|----------|-------------|----------------------|
| **Open-R1 (distill + GRPO)** | Build/study a reasoning model on **verifiable** tasks (math, code); want an open, converging recipe | Heavy: GPUs, vLLM, multi-stage; research repo in flux |
| **Distill-only (SFT on traces)** | You just want a cheaper reasoner and have/can buy R1 traces | No RL gains beyond the teacher; capped by trace quality (this *is* Open-R1's stage 1) |
| **Prompted [chain-of-thought](chain-of-thought.ipynb)** | Quick reasoning boost, no training budget | No weight changes; limited vs a trained reasoner; pay the CoT tokens every call |
| **[DPO / RLHF](trl-rlhf-dpo.ipynb)** | Align on **preferences/taste** (helpfulness, style) — no verifiable answer | Needs preference data or a reward model; not aimed at verifiable reasoning |
| **Off-the-shelf reasoning API** (o-series, R1 hosted) | You want results, not a training pipeline | No control/customization; data leaves your box; per-token cost |
| **TRL `GRPOTrainer` directly** | You want GRPO but your own data/rewards, not R1 reproduction | You build the rewards, configs, eval that Open-R1 already curates |

**Rule of thumb:** verifiable task + want to train → distill first, then GRPO via Open-R1. Preference/taste alignment → DPO/RLHF. No training budget → prompt CoT or call a hosted reasoner.

## 8. Resources

- **Open-R1 repo** — <https://github.com/huggingface/open-r1> (code, recipes, install; the source of truth — pin a commit).
- **Open-R1 launch / update blog posts** — <https://huggingface.co/blog/open-r1> (the plan and progress reports for the reproduction).
- **DeepSeek-R1 paper** — <https://arxiv.org/abs/2501.12948> (the method Open-R1 reproduces: R1-Zero, cold-start SFT, GRPO).
- **TRL `GRPOTrainer` docs** — <https://huggingface.co/docs/trl/main/en/grpo_trainer> (the trainer Open-R1 wraps; reward-func signature, config).
- **`math_verify`** — <https://github.com/huggingface/Math-Verify> (the verifier behind `accuracy_reward`).
- **OpenR1-Math-220k dataset** — <https://huggingface.co/datasets/open-r1/OpenR1-Math-220k> (the distillation traces).
- **Related notebooks:** [TRL / RLHF / DPO](trl-rlhf-dpo.ipynb), [chain-of-thought](chain-of-thought.ipynb), [vLLM](vllm.ipynb).